# RAG + LLM Integration 

This notebook implements the Retrieval-Augmented Generation (RAG) component
for the Dietary Constraint & Inventory-Aware Chef Agent project.


## Goal
The goal of this notebook is to:
1. Load recipe JSON data
2. Build a lightweight retrieval pipeline
3. Connect user queries to the RAG system

In [4]:
import json
from pathlib import Path

DATA_DIR = Path("italian_recipes_json")
assert DATA_DIR.exists(), f"Cannot find {DATA_DIR.resolve()}"

files = sorted(DATA_DIR.glob("*.json"))
print("✅ json files:", len(files), "example:", files[0].name if files else None)

recipes = []
for p in files:
    with open(p, "r", encoding="utf-8") as f:
        recipes.append(json.load(f))

print("✅ loaded recipes:", len(recipes))
print("✅ sample keys:", list(recipes[0].keys()) if recipes else None)

✅ json files: 21 example: budino_di_ricotta.json
✅ loaded recipes: 21
✅ sample keys: ['meals']


## Step 1 — Load Recipe JSON Data

The recipe files generated from the MealDB pipeline are used as the RAG
knowledge source. Each JSON file contains recipe name and ingredients.

In [11]:
def recipe_to_text(obj: dict) -> str:
    meals = obj.get("meals", [])
    if not meals:
        return ""

    meal = meals[0]

    name = meal.get("strMeal", "unknown")
    instructions = meal.get("strInstructions", "")

    ingredients = []
    for i in range(1, 21):
        ing = meal.get(f"strIngredient{i}")
        if ing and ing.strip():
            ingredients.append(ing.strip())

    return f"Recipe: {name}\nIngredients: {', '.join(ingredients)}\nInstructions: {instructions}"

docs = [recipe_to_text(r) for r in recipes]
print("✅ built docs:", len(docs))
print(docs[0][:400])

✅ built docs: 21
Recipe: Budino Di Ricotta
Ingredients: Ricotta, Eggs, Flour, Sugar, Cinnamon, Lemons, Dark Rum, Icing Sugar
Instructions: Mash the ricotta and beat well with the egg yolks, stir in the flour, sugar, cinnamon, grated lemon rind and the rum and mix well. You can do this in a food processor. Beat the egg whites until stiff, fold in and pour into a buttered and floured 25cm cake tin. Bake in the oven 


## Step 2 — Build Lightweight Retrieval

Instead of full vector embeddings, this version implements a simple
keyword-based retrieval method. This allows quick integration with the
LLM orchestration pipeline before embedding-based retrieval is added.

In [12]:
import re

def rag_retrieve(query: str, k: int = 3):
    q_words = re.findall(r"[a-zA-Z]+", query.lower())
    scored = []
    for i, text in enumerate(docs):
        t = text.lower()
        score = sum(1 for w in q_words if w in t)
        scored.append((score, i))
    scored.sort(reverse=True)
    top = [docs[i] for s, i in scored[:k] if s > 0]
    return top

# quick test
res = rag_retrieve("ricotta dessert bake", k=2)
print("found:", len(res))
print(res[0][:250] if res else "None")

found: 2
Recipe: Spinach & Ricotta Cannelloni
Ingredients: Olive Oil, Garlic, Caster Sugar, Red Wine Vinegar, Chopped Tomatoes, Basil Leaves, Mascarpone, Milk, Parmesan, Mozzarella, Spinach, Parmesan, Ricotta, Nutmeg, Cannellini Beans
Instructions: First make


## Step 3 — Connect Local LLM (Phi-3) with RAG Pipeline

In this step, we integrate a local Large Language Model (Phi-3 Mini via Ollama) with the lightweight RAG retrieval system.

The goal is to allow user queries to be answered using grounded recipe context instead of relying on raw LLM generation.

Pipeline Overview:

User Query → RAG Retrieval → Prompt Construction → Local LLM → Grounded Answer

We use a local model instead of a cloud API to ensure reproducibility and avoid external dependencies.

In [16]:
# Step 3 — Connect Local Phi3 (Ollama) to RAG
import requests

def call_llm(prompt: str) -> str:
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "phi3",
        "prompt": prompt,
        "stream": False
    }
    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    return r.json()["response"]
def build_prompt(query: str, contexts: list[str]) -> str:
    context_text = "\n\n".join(contexts)

    return f"""
You are a cooking assistant.

Answer the question using ONLY the recipe context below.

Context:
{context_text}

User Question:
{query}

Answer:
"""

### User Query Interface

This function represents the main orchestration layer of the system.

Responsibilities:

1. Receive a natural language user query
2. Retrieve top-k relevant recipe documents
3. Build a structured prompt using retrieved context
4. Call the local Phi-3 model
5. Return a grounded response

This module acts as the bridge between the retrieval system and the LLM.

In [17]:
def answer_user_query(query: str, k: int = 2):
    retrieved = rag_retrieve(query, k=k)
    prompt = build_prompt(query, retrieved)
    response = call_llm(prompt)

    return {
        "query": query,
        "contexts": retrieved,
        "answer": response
    }


### Example Query

We test the pipeline with a natural language cooking question.

The system retrieves relevant recipes containing ricotta and generates an answer grounded in the retrieved context.

This demonstrates successful integration of:

- Retrieval (RAG)
- Prompt orchestration
- Local LLM inference

In [18]:
res = answer_user_query("How do I cook ricotta pasta?")
print(res["answer"][:1200])

Unfortunately, you cannot directly make "ricotta pasta" as it is not a specific dish in these recipes. However, if we are to infer your request based on the Spinach & Ricotta Cannelloni and considering that ricotta cheese was used along with spinach for filling, here's how you can create a similar dish:

Firstly, prepare the tomato sauce by heating olive oil in a pan and cooking garlic. Add sugar, vinegar, chopped tomatoes (and seasoning), then simmer it to thicken with occasional stirring for about 20 minutes until you achieve your desired consistency – this is essential before layering the pasta shells as per Spinach & Ricotta Cannelloni recipe.

Next, prepare a spinach-ricotta filling by wilting and squeezing out water from fresh spinach in boiling water (you may need to do this in batches), then roughly chopping it afterward along with 100g Parmesan cheese and ricotta; season well with salt, pepper, and nutmeg as instructed.

For the pasta shells – Cannelloni are used here but you 

##Step 3.2 RAG + Llama-3 Cooking Assistant (Implementation Overview)

This function connects our RAG pipeline to the **Meta Llama-3-8B-Instruct** model through the Hugging Face Inference API.

`call_llm()` sends the constructed prompt as a chat request and returns the generated response.  
We use a low temperature (0.2) to encourage more stable, grounded answers based on the retrieved recipe context.


In [ ]:
from huggingface_hub import InferenceClient

HF_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
HF_TOKEN = "hf_yourtoken"

_client = None

def call_llm(prompt: str) -> str:
    global _client
    if _client is None:
        _client = InferenceClient(model=HF_MODEL, token=HF_TOKEN)

    resp = _client.chat_completion(
        messages=[
            {"role": "system", "content": "You are a cooking assistant."},
            {"role": "user", "content": prompt},
        ],
        max_tokens=400,
        temperature=0.2,
    )
    return resp.choices[0].message["content"]

### Prompt Construction

This function builds a grounded prompt by combining the retrieved recipe context with the user’s query.  
It explicitly instructs the LLM to answer using only the provided recipes, helping reduce hallucinations and keep responses aligned with the retrieved data.

In [27]:
def build_prompt(query: str, contexts: list[str]) -> str:
    context_text = "\n\n".join(contexts)
    return f"""
Answer the question using ONLY the recipe context below.
If the answer is not in the context, say "I don't have enough information from the provided recipes."

Context:
{context_text}

User Question:
{query}

Answer:
"""

### Example Query

This cell runs an end-to-end test of the RAG pipeline using a sample cooking question.  
The system retrieves relevant recipes, builds the prompt, and generates a grounded response using Llama-3.

In [30]:
res = answer_user_query("How do I cook the Venetian Duck Ragu pasta?")
print(res["answer"][:800])

Cook the pasta following the pack instructions, then drain, reserving a cup of the pasta water, and add the pasta to the ragu. Stir to coat all the pasta in the sauce and cook for 1 min more, adding a splash of cooking liquid if it looks dry.
